In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

In [ ]:
import sdnq
import gc, torch, gradio as gr
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from diffusers import ModularPipeline

QWEN_ID = "Qwen/Qwen3-VL-4B-Instruct"
KLEIN_ID = "OzzyGT/Flux_3_Image_Klein_sdnq_dynamic_4bit"
REF_SIZE = 384  # references size

SYSTEM = """You write captions for a FLUX 3 Imag Klein generator.
The FIRST image is the PRODUCT: it must appear clearly, with its exact shape, color, material and hardware.
The remaining images, if exist, are a MOOD BOARD: borrow their setting, lighting, palette, camera feel, composition, mood and attitude.
EXPLICIT CONTROLS from the user ALWAYS override the mood board.
Always describe all you can see.
Write ONE full scene caption in this order: medium and style, scene, lighting and camera, each subject with colors, materials and position in the frame.
Make it a high-fashion editorial campaign, not a catalog photo. No bullet points, no preamble. Output only the caption."""

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def load_images(product_path, mood_paths, size):
    paths = [product_path] + list(mood_paths or [])
    ims = [Image.open(p).convert("RGB") for p in paths]
    for im in ims:
        im.thumbnail((size, size))
    return ims

# Step 1: Qwen writes the caption proposition
def write_caption(product_path, mood_paths, request, framing, lighting, camera, setting, palette):
    if not product_path:
        raise gr.Error("Upload a product image first.")
    images = load_images(product_path, mood_paths, 1024)

    controls = {"framing": framing, "lighting": lighting, "camera": camera,
                "setting": setting, "palette": palette}
    controls_text = "\n".join(f"- {k}: {v}" for k, v in controls.items() if v and v.strip()) or "None."
    text = (f"Reference images follow in order (the first image is the PRODUCT).\n"
            f"User request: {request}\n"
            f"Explicit controls (override the mood board characteristics):\n{controls_text}\n"
            f"Output the caption.")

    model = Qwen3VLForConditionalGeneration.from_pretrained(QWEN_ID, dtype=torch.float16, device_map="auto")
    processor = AutoProcessor.from_pretrained(QWEN_ID)
    try:
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": [{"type": "image", "image": im} for im in images]
                                        + [{"type": "text", "text": text}]},
        ]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=450, do_sample=True, temperature=0.7)
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
        caption = processor.batch_decode(trimmed, skip_special_tokens=True,
                                         clean_up_tokenization_spaces=False)[0].strip()
    finally:
        del model, processor
        free_gpu()
    return caption

In [ ]:
# (width, height)
SIZES = {
    "2:3 portrait (full-body)": (512, 768),
    "3:4 portrait": (576, 768),
    "4:5 portrait": (640, 800),
    "1:1 square": (640, 640),
    "4:3 landscape": (768, 576),
    "16:9 wide": (768, 448),
}

# Step 2: Klein generates an image
def generate(product_path, mood_paths, caption, use_mood_ref, seed, aspect):
    width, height = SIZES[aspect]
    if not product_path:
        raise gr.Error("Upload a product image first.")
    if not caption or not caption.strip():
        raise gr.Error("The caption is empty. Write or generate a caption first.")
    refs = load_images(product_path, mood_paths[:1] if (use_mood_ref and mood_paths) else [], REF_SIZE)

    blocks = ModularPipeline.from_pretrained(KLEIN_ID).blocks
    blocks.sub_blocks.pop("prompt_upsampler")
    pipe = blocks.init_pipeline(KLEIN_ID)
    pipe.load_components(dtype=torch.bfloat16)
    pipe.to("cuda")

    try:
        free_gpu()
        img = pipe(prompt=caption, image=refs, height=int(height), width=int(width),
                   generator=torch.Generator().manual_seed(int(seed)), output="images")[0]
        img.save(f"/kaggle/working/campaign_seed{int(seed)}.png")
    except Exception as e:
        if "out of memory" in str(e).lower():
            raise gr.Error("Out of GPU memory.")
        raise
    finally:
        del pipe, blocks
        free_gpu()
    return img

In [ ]:
# Interface Gradio
with gr.Blocks(title="Editorial campaign generator") as demo:
    gr.Markdown("### 1. Prompt Genration")
    with gr.Row():
        product = gr.Image(type="filepath", label="Product photo reference")
        moods = gr.File(file_count="multiple", type="filepath", label="Mood board")
    request = gr.Textbox(label="Request", value="Editorial realistic campaign shot, full-body model wearing this product.")
    with gr.Row():
        framing = gr.Dropdown(["", "full-body, head to toe", "three-quarter length", "waist-up", "close-up detail"],
                              value="full-body, head to toe", label="Framing")
        lighting = gr.Textbox(label="Lighting (optional)", placeholder="hard direct flash")
        camera = gr.Textbox(label="Camera (optional)", placeholder="35mm film, low angle")
    with gr.Row():
        setting = gr.Textbox(label="Setting (optional)", placeholder="tall green grass field")
        palette = gr.Textbox(label="Palette (optional)", placeholder=" greens")
    btn_caption = gr.Button("GENERATE PROMPT")

    gr.Markdown("### 2. Edit the prompt, then generate an image")
    caption_box = gr.Textbox(label="Prompt (editable)", lines=10)
    with gr.Row():
        use_mood_ref = gr.Checkbox(label="Also use first mood board image as a reference", value=False)
        seed = gr.Number(value=1, label="Seed", precision=0)
    with gr.Row():
        aspect = gr.Dropdown(
                             ["2:3 portrait (full-body)", "3:4 portrait", "4:5 portrait", "1:1 square", "4:3 landscape", "16:9 wide"],
                             value="2:3 portrait (full-body)", label="Aspect ratio")
    btn_gen = gr.Button("GENERATE IMAGE")
    result = gr.Image(label="Result", type="pil")

    btn_caption.click(write_caption, [product, moods, request, framing, lighting, camera, setting, palette], caption_box)
    btn_gen.click(generate, [product, moods, caption_box, use_mood_ref, seed, aspect], result)

demo.launch(share=True, debug=True)